#### Задача 1 (10 баллов). 

Попробуем себя в решении задачи определения темы текста. Будем считать, что два текста похожи по теме, если у них больше общих слов (только не предлогов с союзами), чем у других текстов. У нашей программы для определения темы будет несколько готовых текстов (достаточно больших!) с уже известной темой в базе: выберите тексты (и темы) самостоятельно, 5-6 будет достаточно. 

Что должна делать программа? При запуске вы ей сообщаете название нового файла с текстом, который нужно классифицировать, она его открывает, обрабатывает и сравнивает с текстами в своей базе. С которым из текстов оказалось больше всего общих слов, того и тема! Очевидно, вам понадобится какие-то слова из текстов отбрасывать (подумайте, каким образом это сделать - здесь на самом деле несколько вариантов концепций), а еще лемматизировать или хотя бы использовать стемминг. 

Когда будете сдавать это задание, пожалуйста, пришлите и файлы с текстами. И имейте в виду, если тексты будут вставлены прямо в код и слишком короткие, я задачу засчитаю только вполовину. 

Напоминаю, как открываются файлы:

```
with open('путь к файлу - пишите прямые слеши', 'r', encoding='utf-8') as f:
    text = f.read() # все содержимое вашего файла считается в одну длинную строку
```

Настоятельно советую оформить код хотя бы в функции. 

In [1]:
import os
import spacy
from nltk.corpus import stopwords

In [2]:
nlp = spacy.load('ru_core_news_md')
stopwords = set(stopwords.words('russian'))

In [3]:
def process(text):
    '''Эта функция для лемматизации, удаления стоп-слов и приведения к нижнему регистру'''
    text = text.lower().split() # убираем стоп-слова
    nostopwords = ' '.join([word for word in text if word.lower() not in stopwords])
    doc = nlp(nostopwords)
    lemmatisedtext = ' '.join([token.lemma_ for token in doc if not token.is_punct])  # and not token.is_space
    return lemmatisedtext


def similarity(usertext, basetext):
    '''Эта функция для сравнения нового текста с текстами из базы'''
    userwords = set(usertext)
    basewords = set(basetext)
    common = userwords.intersection(basewords)
    return len(common)
    

def loadtextsdb():
    '''Загрузка текстов из папки с базой текстов'''
    textsbase = {}
    for filename in os.listdir(r'D:\Компьютерная лингвистика\дз 2\texts'):
        with open(os.path.join(r'D:\Компьютерная лингвистика\дз 2\texts', filename), 'r', encoding='utf-8') as file:
            textsbase[filename] = file.read()
    return textsbase

def getusertext():
    '''Получаем файл с новым текстом'''
    filename = input('Введите название нового файла с текстом: ') # usertext.txt
    with open(filename, 'r', encoding="utf8") as file:
        newtext = file.read()
    return newtext


In [4]:
textsdb = loadtextsdb() # загружаем базу, и новый текст
newtext = getusertext() # usertext.txt ! usertext2.txt - оба текста в жанре фантастика (в первом файле часть текста из файла sci-fi, во втором отрывок из другой книги того же жанра)
newtextprocessed = process(newtext)

maxsimilarity = 0 # ищем текст в базе с наибольшей схожестью, и предварительно его тоже проводим через process()
for k, v in textsdb.items():
    basetextprocessed = process(v)
    similar = similarity(newtextprocessed, basetextprocessed)
    if similar > maxsimilarity:
        maxsimilarity = similar
        proposedtheme = os.path.splitext(k)[0] # имя файла = тема
print("Тема этого текста скорее всего:", proposedtheme)

Тема этого текста скорее всего: sci-fi


#### Задача 2 (10 баллов). 

Некоторые предлоги в русском языке могут управлять разными падежами (например, "я еду в Лондон" vs "я живу в Лондоне"). Давайте проанализируем эти предлоги и их падежи. Необходимо:

- составить список таких предлогов (РГ-80 вам в помощь)
- взять достаточно большой текст (можно большое художественное произведение)
- сделать морфоразбор этого текста
- Посчитать, как часто и какие падежи встречаются у слова, идущего после предлога.

Примечания: во-первых, имейте в виду, что иногда после предлога могут идти самые неожиданные вещи: "я что, должен ехать на, черт побери, северный полюс?". Во-вторых, неплохо бы учитывать отсутствие пунктуации (конечно, в норме, как нам кажется, предлог обязательно требует зависимое, но! "да иди ты на!") Эти штуки можно отсеять, если просто учитывать только заранее определенные падежи, а не считать все, какие встретились (так и None можно огрести). 

Если будете использовать RNNMorph, возможно, понадобится регулярное выражение и немного терпения. 

In [5]:
from collections import defaultdict
from nltk.tokenize import word_tokenize
from pymorphy2 import MorphAnalyzer

In [6]:
# с rnnmorph у меня была какая-то беда... несколько раз переустанавливала, но пока так и не получилось сделать так, чтобы хоть что-то заработало

morph = MorphAnalyzer()

with open(r'D:\Компьютерная лингвистика\дз 2\каллисто.txt', 'r', encoding='utf-8') as f:
    text8 = f.read()

tokens = word_tokenize(text8.lower())
prepscases = defaultdict(lambda: defaultdict(int)) 
 
for i in range(len(tokens)-1): 
    if morph.parse(tokens[i])[0].tag.POS == 'PREP': 
        caseoftheword = morph.parse(tokens[i+1])[0].tag.case 
        if caseoftheword is not None: 
            prepscases[tokens[i]][caseoftheword] += 1 
 

prepscases = {prep: cases for prep, cases in prepscases.items() if len(cases) >= 2} # выбираю только те, где больше 2х падежей
 

for preposition, cases in prepscases.items():
    print(f'\nС предлогом "{preposition}" встречаются падежи:')
    for case, count in cases.items():
        print(f'{case}: {count} раз')       



С предлогом "на" встречаются падежи:
loct: 921 раз
accs: 607 раз
gent: 366 раз
ablt: 13 раз
datv: 35 раз
nomn: 428 раз
loc2: 24 раз

С предлогом "к" встречаются падежи:
datv: 649 раз
accs: 35 раз
loct: 76 раз
gent: 70 раз
loc2: 5 раз
nomn: 7 раз
ablt: 10 раз

С предлогом "с" встречаются падежи:
accs: 29 раз
ablt: 882 раз
gent: 319 раз
datv: 81 раз
nomn: 10 раз
loct: 25 раз
loc2: 1 раз

С предлогом "в" встречаются падежи:
loct: 1084 раз
datv: 29 раз
accs: 672 раз
nomn: 353 раз
gent: 345 раз
ablt: 9 раз
loc2: 23 раз

С предлогом "по" встречаются падежи:
loct: 43 раз
datv: 334 раз
gent: 65 раз
accs: 43 раз
nomn: 8 раз
ablt: 9 раз
loc2: 16 раз
gen2: 1 раз

С предлогом "за" встречаются падежи:
ablt: 156 раз
accs: 94 раз
loct: 12 раз
nomn: 57 раз
gent: 25 раз
datv: 51 раз

С предлогом "у" встречаются падежи:
gent: 326 раз
accs: 89 раз
nomn: 15 раз
loct: 13 раз

С предлогом "от" встречаются падежи:
gent: 363 раз
accs: 26 раз
nomn: 17 раз
datv: 1 раз
ablt: 1 раз
loct: 7 раз

С предлогом "из" 

#### Задача 3 (5 баллов). 

Представим, что у вас есть файл с разборами conllu (можете взять любой, например, [тут](https://github.com/dialogue-evaluation/GramEval2020)). Нужно просмотреть все примеры предложений с тегом dislocated и тегом discourse: напишите скрипт, который будет читать файл, находить все такие предложения и печатать: 1) сам текст предложения 2) слово, имеющее искомый тег. Если тег не был найден в файле, нужно об этом сообщить. Постарайтесь оформить вывод таким образом, чтобы это было удобно читать. 

In [7]:
import pyconll

#token.id, token.form, token.lemma, token.upos, token.feats, token.head, token.deprel, token.misc

textcon = pyconll.load_from_file('GramEval2020-Taiga-social-train.conllu')  #  GramEval2020-Taiga-social-train.conllu    GramEval2020-RuEval2017-social-dev.conllu

foundit = False
for sentence in textcon:
    for token in sentence:
        if 'dislocated' in token.deprel or 'discourse' in token.deprel:
            print(f'Найден тег dislocated/discourse в предложении: {sentence.text}')
            print(f'Токен с таким тегом: {token.form}, тег - {token.deprel}\n')
            foundit = True            
if not foundit:
    print('Dislocated или discourse не найдены')

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: ), тег - discourse

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: @, тег - discourse

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: что, тег - dislocated

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: то, тег - discourse

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: "))), тег - discourse

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: :), тег - discourse

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: @yabloko, тег - dislocated

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: )))), тег - discourse

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: гггг, тег - discourse

Найден тег dislocated/discourse в предложении: None
Токен с таким тегом: @yabloko, тег - dislocated

Найден тег dislocated/discourse в предло

#### Задача 4 (5 баллов).

Возьмите любой достаточно длинный (лучше новостной) текст. Любым известным инструментом извлеките именованные сущности из этого текста и выведите их списком по категориям (т.е. персоны вместе, локации вместе, организации вместе). 

In [8]:
nlp = spacy.load('ru_core_news_md')
with open(r'D:\Компьютерная лингвистика\дз 2\recognition.txt', 'r', encoding='utf-8') as f:
    text2 = f.read() 
doc = nlp(text2)
entities = {'LOC': [], 'ORG': [], 'PER': [],}
for ent in doc.ents:
    if ent.label_ in entities:
        entities[ent.label_].append(ent.text)

print(f'Persons: {entities["PER"]}\nOrganisations: {entities["ORG"]}\nLocations: {entities["LOC"]}')    

Persons: ['Джейсона', 'Нил Дракманн', 'Дракманна']
Organisations: ['HoYoverse (miHoYo)', 'Honkai', 'Zenless Zone Zero', 'ЗБТ', 'Genshin Impact', 'Honkai', 'Take-Two', 'PlayStation', 'Xbox', 'Xbox', 'ПК', 'Take-Two', 'Rockstar Games', 'Naughty Dog', 'The Last of Us', 'Kotaku', 'PlayStation', 'Lesta Games', 'Forbes']
Locations: ['Нью-Эриду', 'Леонида', 'Вайс-Сити', 'Вайс-Сити', 'Лусию']
